# Train Apprentice's SQL judge (Kev-0.8B fine-tune)

Fine-tunes [Kev-0.8B](https://github.com/jaredpalmer/kev) (Apache-2.0) to decide whether a SQL query answers a question.
Training data comes from `bench/judge_data.py` in [Ayush-e4/apprentice](https://github.com/Ayush-e4/apprentice): labels are exact, from comparing each query's result with the gold query's result.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.

Steps: install Kev → download data → score the released model → fine-tune → score again → save → (optional) serve over a public URL.

In [ ]:
REPO = 'Ayush-e4/apprentice'
BRANCH = 'main'
BASE = 'Qwen/Qwen3.5-0.8B-Base'
BASE_REVISION = 'dc7cdfe2ee4154fa7e30f5b51ca41bfa40174e68'  # the base Kev-0.8B was trained on
INIT_FROM = 'jaredpalmer/kev-0.8b'
OUT = 'runs/sql-judge'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Install Kev from source (it is not on PyPI; `pip install kev` is a different package)
!git clone -q --depth 1 https://github.com/jaredpalmer/kev.git
%cd kev
!pip install -q -e '.[serve]'
# Kev pins torch<2.9, which downgrades Colab's torch. Colab's torchvision is built for the newer torch and
# breaks `import transformers` ("operator torchvision::nms does not exist"), and its torchao is older than
# peft accepts ("only versions above 0.16.0 are supported"). Kev uses neither.
!pip uninstall -y -q torchvision torchaudio torchao
!python -c "import kev, torch, transformers, peft; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# Download the labelled data
raw = f'https://raw.githubusercontent.com/{REPO}/{BRANCH}/data/judge'
!mkdir -p data && wget -q {raw}/train.jsonl -O data/train.jsonl && wget -q {raw}/heldout.jsonl -O data/heldout.jsonl
!wc -l data/train.jsonl data/heldout.jsonl

In [ ]:
# Baseline: the released Kev-0.8B on our held-out SQL questions
!python -m kev.benchmark --run {INIT_FROM} --data data/heldout.jsonl --device cuda --out runs/base-eval
import os
assert os.path.exists('runs/base-eval/report.json'), 'baseline failed: read the error above'

In [ ]:
# Fine-tune: start from the released adapter (keeps what Kev already knows), small learning rate.
# fp32 because the T4 has no bf16 support.
!python -m kev.train --data data/train.jsonl --base {BASE} --base_revision {BASE_REVISION} \
    --init_from {INIT_FROM} --epochs 2 --lr 2e-5 --batch 4 --accum 2 --dtype fp32 \
    --checkpointing 1 --device cuda --out {OUT}
import os
assert os.path.exists(f'{OUT}/head.pt'), 'training failed: read the error above'

In [ ]:
# After: the fine-tuned judge on the same held-out questions
!python -m kev.benchmark --run {OUT} --data data/heldout.jsonl --device cuda --out runs/tuned-eval
import os
assert os.path.exists('runs/tuned-eval/report.json'), 'evaluation failed: read the error above'

In [ ]:
# Side by side
import json
for name in ('base-eval', 'tuned-eval'):
    report = json.load(open(f'runs/{name}/report.json'))
    print(f'== {name} ==')
    print(json.dumps(report.get('objective', report), indent=2)[:1500])

In [ ]:
# Save the trained judge to your computer
!cd runs && zip -qr sql-judge.zip sql-judge base-eval tuned-eval
from google.colab import files
files.download('runs/sql-judge.zip')

## Optional: serve the judge from Colab

Runs the trained judge here and exposes it on a temporary public HTTPS URL (Cloudflare quick tunnel), protected by an API key.
Put the printed values in Apprentice's `.env` as `JUDGE_BASE_URL` and `JUDGE_API_KEY`. The URL stops working when this Colab session ends.

In [ ]:
import os, re, secrets, subprocess, time
key = secrets.token_hex(24)
env = {**os.environ, 'KEV_API_KEY': key}
server = subprocess.Popen(['python', '-m', 'kev.serve', '--run', OUT, '--port', '8009', '--device', 'cuda'],
                          env=env, stdout=open('serve.log', 'w'), stderr=subprocess.STDOUT)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8009'],
                          stdout=open('tunnel.log', 'w'), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(2)
    found = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel.log').read())
    if found:
        url = found.group(0)
        break
print('JUDGE_BASE_URL=' + str(url))
print('JUDGE_API_KEY=' + key)
print('JUDGE_MODEL=kev-sql-judge')